# 🚀  **Ultra**

> A streamlined backtesting environment for rapid model iteration and evaluation.

---

## Overview

**Ultra** is designed to efficiently backtest:

- 📊 **Existing models** — validate current production models against historical data
- 🔧 **Model modifications** — test tweaks and improvements before deployment
- 🧪 **New variables** — evaluate feature additions and their predictive power

---
| Feature | Description |
|---------|-------------|
| ⚡ **Production-level accuracy** | Uses production datasets, methods, and node selection that closely mirror the live environment |
| 🎯 **Compact node selection** | For initial backtests, focuses on a rotating subset — selecting a few nodes per day while ensuring the full training set covers all unique nodes |
| 📈 **Fast visualization** | Leverages Kubernetes to run each backtest within minutes, enabling rapid visualization of results |
| 🔄 **Two-level backtest** | Runs an initial backtest on a smaller subset first; if significant improvement is observed, it escalates to a full backtest |
---

## Testing Content

**Guide:** We benchmark all results against the **Fourier model**, keeping every variable constant — testing window, node selection, and all other parameters — except for the specific model structure or variable being tested.

Each experiment is graded as follows:

| Grade | Meaning |
|:-----:|---------|
| 🟢 **S** | Significant improvement over production |
| 🔵 **A** | Slight improvement, but not significant |
| ⚪ **B** | Nearly identical to production |
| 🟡 **C** | Slightly worse than production |
| 🔴 **F** | Significantly worse than production (failed) |

---

### Model Tweaks

1. Fourier with training set, no shuffle — **B**
2. Fourier with larger batch size: **C**
2. CNN + Fourier: **F**
3. Fourier + CNN: **F**
4. pure CNN: **F**
5. GRU: **A**
6. GRU with updated parameters(hidden size: 16, learning rate: 0.001, out= mean(), better than 5th): **A**
7. GRU with slack and congestion trained separately (no panel): **B**
8. Transformer: **C** (similar to GRU)

In [2]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '/var/www/python/Qingcheng/Darwin')
sys.path.append('/var/www/python/Prod/nighthawk/')
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
from nighthawk.util import bigquery_functions, connections
from nighthawk.models.valuation import node_price_predictor
from google.cloud import bigquery, storage
warnings.filterwarnings("ignore")
from datetime import datetime
import sys, os, inspect, shutil, importlib
sys.path.insert(0, '/var/www/python/Qingcheng/Fourier')
import pandas as pd
import numpy as np
from nighthawk.util import bigquery_functions, connections, sql_functions, dataframe_functions
from nighthawk.data.pipeline.var_handler import loadwindgen_vh, wind_vh
from nighthawk.data.pipeline.common_functions import wind
from google.cloud import bigquery
import utils_fourier.ve_portfolio_constructor_fourier as ve_portfolio_constructor_fourier
from common_functions import get_scale_factor
from nighthawk.data.network.node import Node
sys.path.insert(0, '/var/www/python/Qingcheng/QCTest/Ultra')
from custom import get_metrics, eval_valuation_model, run_in_kubernetes, fourier_port, simulate_total_ftp, select_unique_nodes_across_dates, pnl_metrics, run_valuation_backtest, load_from_ve_strategy
from model import GRU_framework_means_update_para, GRU_framework_means_print, GRU_feature_importance, NN_training_module_no_shuffle, Transformer_framework, GRU_SHAP, GRU_framework_24step
import math
from dataset import all_tables


Here the job starts!
2026-05-20 22:15:09.544196


In [3]:
select_unique_nodes_across_dates('2026-02-13','2026-05-20',10,True) 

,dt,node_num
0,2026-02-13,1673
1,2026-02-13,1163
2,2026-02-13,200
3,2026-02-13,1027
4,2026-02-13,857
...,...,...
965,2026-05-20,1735
966,2026-05-20,359
967,2026-05-20,1725
968,2026-05-20,1546


In [4]:
# train slack and congestion separately and combine them together 
# da and rt slack

result, valuation= run_valuation_backtest(Transformer_framework,'2026-02-13_2026-05-20_node_selection.csv',10000)
result.to_csv(f'/var/www/python/Qingcheng/WFiles/Ultra/model_val/Transformer_val_{datetime.today().strftime("%Y-%m-%d")}.csv')

  record_id 50800
We gonna upload data into cloud sql!
data upload done!


KeyboardInterrupt: 

In [5]:
dataset = load_from_ve_strategy(50793)
dataset.to_csv(f'/var/www/python/Qingcheng/WFiles/Ultra/model_val/Transformer_val_{datetime.today().strftime("%Y-%m-%d")}.csv')

In [6]:
get_metrics(dataset)

,Prod
da_total_ME,-4.26
da_total_MSE,2324.74
da_total_MAE,20.87
da_total_RMSE,48.22
da_total_MAPE,0.81
da_total_RMSE_first,27.40
rt_total_ME,-0.91
rt_total_MSE,3635.02
rt_total_MAE,27.20
rt_total_RMSE,60.29


In [8]:
# da and rt congestion 
result_fourier, valuation_fourier= run_valuation_backtest(GRU_SHAP,"2026-02-07_2026-05-05_node_selection.csv",2)
result_fourier.to_csv(f'/var/www/python/Qingcheng/WFiles/Ultra/model_val/GRU_SHAP_{datetime.today().strftime("%Y-%m-%d")}.csv')

  record_id 50717
We gonna upload data into cloud sql!
data upload done!


sh: 0: getcwd() failed: No such file or directory


docker: 'docker rmi' requires at least 1 argument

Usage:  docker rmi [OPTIONS] IMAGE [IMAGE...]

See 'docker rmi --help' for more information

Login Succeeded
 
WARNING! Your credentials are stored unencrypted in '/home/qingcheng/.docker/config.json'.
Configure a credential helper to remove this warning. See
https://docs.docker.com/go/credential-store/


Using default tag: latest
latest: Pulling from movetocloud-999/fourier/ve_2024_nn
Digest: sha256:22c06995af4e5d82cc39deb8db214c0f8193dd36eef898519bacea228b671c5a
Status: Image is up to date for us-central1-docker.pkg.dev/movetocloud-999/fourier/ve_2024_nn:latest
us-central1-docker.pkg.dev/movetocloud-999/fourier/ve_2024_nn:latest
 
 #0 building with "default" instance using docker driver

#1 [internal] load build definition from Dockerfile
#1 transferring dockerfile: 382B done
#1 WARN: ConsistentInstructionCasing: Command 'workdir' should match the case of the command majority (uppercase) (line 3)
#1 WARN: ConsistentInstructionCasing:

NotFound: 404 POST https://bigquery.googleapis.com/bigquery/v2/projects/movetocloud-999/jobs?prettyPrint=false: Not found: Table movetocloud-999:backtest.SPPVEpredictionRecord_50717 was not found in location US

Location: None
Job ID: 6bb519a8-7915-4631-bf88-061e9ce40f15


In [ ]:
50063

hidden size: 32, learning rate: 0.01, out = last one 
da_total_ME      -5.03    
da_total_MSE    411.89
da_total_MAE     14.98
da_total_RMSE    20.29
rt_total_ME      -2.79
rt_total_MSE   3760.65
rt_total_MAE     25.75
rt_total_RMSE    61.32

hidden size: 16, learning rate: 0.001, out = mean(), refer to 50064
da_total_ME      -5.39
da_total_MSE    377.39
da_total_MAE     14.62
da_total_RMSE    19.43
rt_total_ME      -2.06
rt_total_MSE   3746.30
rt_total_MAE     25.88
rt_total_RMSE    61.21

Full backtest for 273 for GRU on updated parameter
da_total_ME      -5.73
da_total_MSE   2254.18
da_total_MAE     21.69
da_total_RMSE     47.4
rt_total_ME      -0.08
rt_total_MSE   3434.03
rt_total_MAE     26.89
rt_total_RMSE    58.60

Full backtest for 273 for Fourier on updated parameter 
da_total_ME      -3.04
da_total_MSE   3771.45
da_total_MAE     20.53
da_total_RMSE    61.41
rt_total_ME       2.80
rt_total_MSE   7668.83
rt_total_MAE     31.37
rt_total_RMSE    87.57

Full backtest for 273 for GRU on da and rt slack trained separately
da_slack_ME            -4.69
da_slack_MSE         1936.34
da_slack_MAE           19.08
da_slack_RMSE          44.00
da_slack_MAPE          71.85
da_slack_RMSE_first    33.44
rt_slack_ME             0.67
rt_slack_MSE         2164.61
rt_slack_MAE           21.31
rt_slack_RMSE          46.53
rt_slack_MAPE         106.57
rt_slack_RMSE_first    19.46

Full backtest for 273 for GRU on da and rt congestion trained separately
da_congestion_ME            -1.40
da_congestion_MSE          613.28
da_congestion_MAE           13.42
da_congestion_RMSE          24.76
da_congestion_MAPE         139.87
da_congestion_RMSE_first    15.23
rt_congestion_ME            -1.21
rt_congestion_MSE         1449.39
rt_congestion_MAE           18.43
rt_congestion_RMSE          38.07
rt_congestion_MAPE         164.97
rt_congestion_RMSE_first    42.59

Full backtest for 273 for GRU total 5/5
da_total_ME            -5.02
da_total_MSE         2201.06
da_total_MAE           21.56
da_total_RMSE          46.92
da_total_MAPE          86.51
da_total_RMSE_first    33.22
rt_total_ME             0.06
rt_total_MSE         3560.03
rt_total_MAE           27.01
rt_total_RMSE          59.67
rt_total_MAPE         131.41
rt_total_RMSE_first    32.05

Transformer for 267 total 5/7
da_total_ME	-6.31
da_total_MSE	2448.33
da_total_MAE	20.74
da_total_RMSE	49.48
da_total_MAPE	76.48
da_total_RMSE_first	31.84
rt_total_ME	-2.39
rt_total_MSE	3672.10
rt_total_MAE	26.73
rt_total_RMSE	60.60
rt_total_MAPE	122.17
rt_total_RMSE_first	31.09

NN for 273 5/7
da_total_ME            -3.30
da_total_MSE         3525.79
da_total_MAE           20.30
da_total_RMSE          59.38
da_total_MAPE          94.83
da_total_RMSE_first    19.38
rt_total_ME             1.66
rt_total_MSE         9937.61
rt_total_MAE           32.57
rt_total_RMSE          99.69
rt_total_MAPE         174.96
rt_total_RMSE_first    32.52

GRU for 1312 nodes 
da_total_ME	-4.72
da_total_MSE	2138.80
da_total_MAE	21.73
da_total_RMSE	46.25
da_total_MAPE	0.89
da_total_RMSE_first	35.59
rt_total_ME	0.24
rt_total_MSE	3551.71
rt_total_MAE	27.48
rt_total_RMSE	59.60
rt_total_MAPE	1.32
rt_total_RMSE_first	54.35
da_rt_spread_ME	25.27
da_rt_first_spread_ME	22.37


NN for 1312 nodes
da_total_ME	-1.66
da_total_MSE	3703.37
da_total_MAE	20.56
da_total_RMSE	60.86
da_total_MAPE	0.95
da_total_RMSE_first	25.39
rt_total_ME	2.88
rt_total_MSE	8987.66
rt_total_MAE	32.51
rt_total_RMSE	94.80
rt_total_MAPE	1.68
rt_total_RMSE_first	55.31
da_rt_spread_ME	31.47
da_rt_first_spread_ME	23.24


# GRU for 273 training 24 steps
# da_total_ME              -3.89
# da_total_MSE           2135.21
# da_total_MAE             20.24
# da_total_RMSE            46.21
# da_total_MAPE             0.80
# da_total_RMSE_first      41.91
# rt_total_ME               2.68
# rt_total_MSE           3509.46
# rt_total_MAE             28.49
# rt_total_RMSE            59.24
# rt_total_MAPE             1.46
# rt_total_RMSE_first      59.97


GRU 1312 with updated parameters 5/11
da_total_ME	-5.82
da_total_MSE	1908.64
da_total_MAE	18.96
da_total_RMSE	43.69
da_total_MAPE	0.71
da_total_RMSE_first	25.92
rt_total_ME	-5.19
rt_total_MSE	3348.98
rt_total_MAE	23.97
rt_total_RMSE	57.87
rt_total_MAPE	1.00
rt_total_RMSE_first	66.00
da_rt_spread_ME	23.40
da_rt_first_spread_ME	21.77

Transformer 1312 5/11
da_total_ME	-4.26
da_total_MSE	2324.74
da_total_MAE	20.87
da_total_RMSE	48.22
da_total_MAPE	0.81
da_total_RMSE_first	27.40
rt_total_ME	-0.91
rt_total_MSE	3635.02
rt_total_MAE	27.20
rt_total_RMSE	60.29
rt_total_MAPE	1.26
rt_total_RMSE_first	64.53
da_rt_spread_ME	25.01
da_rt_first_spread_ME	23.42

SyntaxError: invalid syntax (1759962447.py, line 3)

# Data Training File Analysis

In [6]:
import pandas as pd
nodes = pd.read_csv('/var/www/python/Qingcheng/WFiles/Ultra/node_selection/2026-02-13_2026-05-11_node_selection.csv')
display(nodes[200:220])

,Unnamed: 0,dt,node_num
200,200,2026-02-26,1384
201,201,2026-02-26,420
202,202,2026-02-26,1255
203,203,2026-02-26,463
204,204,2026-02-26,700
205,205,2026-02-26,1164
206,206,2026-02-26,1359
207,207,2026-02-26,113
208,208,2026-02-26,200
209,209,2026-02-26,1429


In [7]:
import pandas as pd

df144  = pd.read_csv('gs://ve_fourier/production/SPP/training/221_2026-02-27.csv')
df1666 = pd.read_csv('gs://ve_fourier/production/SPP/training/991_2026-02-27.csv')

cols931  = set(df144.columns)
cols1683 = set(df1666.columns)

only_931  = sorted(cols931 - cols1683)
only_1683 = sorted(cols1683 - cols931)
shared    = sorted(cols931 & cols1683)

print(f'931 total cols:  {len(cols931)}')
print(f'1683 total cols: {len(cols1683)}')
print(f'Shared: {len(shared)}')
print(f'\nOnly in 931  ({len(only_931)}):  {only_931}')
print(f'\nOnly in 1683 ({len(only_1683)}): {only_1683}')
print(f'\nAll 931 columns ({len(cols931)}):')
for c in sorted(df144.columns):
    print(' ', c)


931 total cols:  222
1683 total cols: 266
Shared: 115

Only in 931  (107):  ['BackwardRampNoSlope1_indn_spp_zonal_load_actual_a', 'BackwardRampNoSlope1_indn_spp_zonal_load_forecast_f', 'BackwardRampNoSlope1_south_spp_broadzonal_load_actual_a', 'BackwardRampNoSlope1_south_spp_broadzonal_load_forecast_f', 'ForwardRampNoSlope1_indn_spp_zonal_load_actual_a', 'ForwardRampNoSlope1_indn_spp_zonal_load_forecast_f', 'ForwardRampNoSlope1_south_spp_broadzonal_load_actual_a', 'ForwardRampNoSlope1_south_spp_broadzonal_load_forecast_f', 'Perc_30D_indn_spp_zonal_load_actual_a', 'Perc_30D_indn_spp_zonal_load_forecast_f', 'Perc_30D_rz4_spp_res_zonal_load_wind_diff_actual_a', 'Perc_30D_rz4_spp_res_zonal_load_wind_diff_forecast_f', 'SPPResZone4SimHr_da_congestion', 'SPPResZone4SimHr_da_total', 'SPPResZone4SimHr_rt_congestion', 'SPPResZone4SimHr_rt_total', 'altw_miso_zonal_load_actual_a', 'altw_miso_zonal_load_forecast_f', 'ammo_miso_zonal_load_actual_a', 'ammo_miso_zonal_load_forecast_f', 'central_miso_b

In [14]:
df144.columns[150:200]

Index(['kansascity_mo_79_temperature_humidity_index',
       'gfs_wspd80_1016_FrontierWindpow_actual_a',
       'gfs_wspd80_1016_FrontierWindpow_forecast_f',
       'gfs_wspd80_1081_DiamondTrailWin_actual_a',
       'gfs_wspd80_1081_DiamondTrailWin_forecast_f',
       'gfs_wspd80_1120_IrishCreekWind_actual_a',
       'gfs_wspd80_1120_IrishCreekWind_forecast_f',
       'gfs_wspd80_1139_Milligan1WindFa_actual_a',
       'gfs_wspd80_1139_Milligan1WindFa_forecast_f',
       'gfs_wspd80_280_PioneerPrairieW_actual_a',
       'gfs_wspd80_280_PioneerPrairieW_forecast_f',
       'gfs_wspd80_348_SmokyHillsWindP_actual_a',
       'gfs_wspd80_348_SmokyHillsWindP_forecast_f',
       'gfs_wspd80_462_LakefieldWindPr_actual_a',
       'gfs_wspd80_462_LakefieldWindPr_forecast_f',
       'gfs_wspd80_584_Spearville3LLC_actual_a',
       'gfs_wspd80_584_Spearville3LLC_forecast_f',
       'gfs_wspd80_851_RockCreekWindPr_actual_a',
       'gfs_wspd80_851_RockCreekWindPr_forecast_f',
       'gfs_wspd80_883_B

In [20]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 1. Isolate numerical features
X = df144.select_dtypes(include=['float64', 'int64']) 

# NEW: Fill missing values with the mean of each column
# (Alternatively, you can use X = X.fillna(0) to fill with zeros)
X = X.fillna(X.mean())
X = X.fillna(0)

# 2. Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Fit PCA
pca = PCA(n_components=20)
pca_result = pca.fit_transform(X_scaled)

df_pca = pd.DataFrame(data=pca_result)
print("PCA Successful! Shape:", df_pca.shape)

PCA Successful! Shape: (17590, 20)


In [26]:
import pandas as pd
import numpy as np

# === 1. Variance explained by each PC ===
variance_ratio = pca.explained_variance_ratio_ * 100

variance_df = pd.DataFrame({
    'Component': [f'PC{i+1}' for i in range(len(variance_ratio))],
    'Variance_Pct': variance_ratio.round(2),
    'Cumulative_Pct': variance_ratio.cumsum().round(2)
})

print("--- PCA Variance Scores ---")
print(variance_df.to_string(index=False))

# === 2. Loadings: how each feature contributes to each PC ===
# Replace `feature_names` with your column list, e.g. X.columns
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(pca.n_components_)],
    index=X.columns
)

print("\n--- PCA Loadings (raw) ---")
print(loadings.round(3))

# === 3. Top contributing features per PC (by absolute loading) ===
print("\n--- Top 5 Contributors per PC ---")
for pc in loadings.columns:
    top = loadings[pc].abs().sort_values(ascending=False).head(5)
    print(f"\n{pc}:")
    for feat, val in top.items():
        # show original sign alongside magnitude
        signed = loadings.loc[feat, pc]
        print(f"  {feat:<30} {signed:+.3f}")

# === 4. Squared loadings as % contribution (sums to 100 per PC) ===
contribution_pct = (loadings ** 2 * 100).round(2)
print("\n--- % Contribution per PC (squared loadings) ---")
print(contribution_pct)

--- PCA Variance Scores ---
Component  Variance_Pct  Cumulative_Pct
      PC1         21.58           21.58
      PC2         14.27           35.85
      PC3         13.44           49.28
      PC4          6.12           55.41
      PC5          4.94           60.35
      PC6          2.76           63.11
      PC7          2.56           65.67
      PC8          2.22           67.89
      PC9          2.00           69.89
     PC10          1.91           71.80
     PC11          1.78           73.59
     PC12          1.36           74.95
     PC13          1.22           76.16
     PC14          1.10           77.26
     PC15          1.00           78.27
     PC16          0.88           79.14
     PC17          0.78           79.92
     PC18          0.76           80.68
     PC19          0.75           81.43
     PC20          0.67           82.10

--- PCA Loadings (raw) ---
                     PC1    PC2    PC3    PC4    PC5    PC6    PC7    PC8  \
hr                 0.032  0

In [28]:
import pandas as pd
import numpy as np

# === Loadings DataFrame ===
# Replace `feature_names` with your column list, e.g. xtrain.columns
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(pca.n_components_)],
    index=X.columns
)

# === Top 10 contributors per PC ===
top_n = 10
print(f"--- Top {top_n} Contributors per PC ---\n")

for pc in loadings.columns:
    # Rank by absolute loading (magnitude = importance, sign = direction)
    ranked = loadings[pc].abs().sort_values(ascending=False).head(top_n)
    
    print(f"{pc} (explains {pca.explained_variance_ratio_[int(pc[2:])-1]*100:.2f}% of variance)")
    print("-" * 60)
    for rank, (feat, abs_val) in enumerate(ranked.items(), 1):
        signed = loadings.loc[feat, pc]
        pct_contrib = (signed ** 2) * 100   # squared loading = % contribution to this PC
        print(f"  {rank:2d}. {feat:<35} {signed:+.4f}  ({pct_contrib:5.2f}%)")
    print()

--- Top 10 Contributors per PC ---

PC1 (explains 21.58% of variance)
------------------------------------------------------------
   1. south_spp_broadzonal_loadwindgen_forecast_f +0.1375  ( 1.89%)
   2. rz4_spp_res_zonal_load_wind_diff_forecast_f +0.1374  ( 1.89%)
   3. south_spp_broadzonal_loadwindgen_actual_a +0.1359  ( 1.85%)
   4. rz4_spp_res_zonal_load_wind_diff_actual_a +0.1339  ( 1.79%)
   5. spp_northsouthdiff_f                -0.1325  ( 1.75%)
   6. spp_northsouthdiff_a                -0.1313  ( 1.72%)
   7. spp_southsppmisodiff_f              +0.1158  ( 1.34%)
   8. spp_southsppmisodiff_a              +0.1107  ( 1.23%)
   9. csws_spp_zonal_load_forecast_f      +0.1085  ( 1.18%)
  10. Perc_30D_rz4_spp_res_zonal_load_wind_diff_forecast_f +0.1077  ( 1.16%)

PC2 (explains 14.27% of variance)
------------------------------------------------------------
   1. spp_south_temperature_degf          +0.1504  ( 2.26%)
   2. wichita_ks_83_temperature_humidity_index +0.1500  ( 2.25%)
   

In [46]:
import pandas as pd

df = pd.read_csv('gs://ve_fourier/production/SPP/training/144_2026-02-13.csv')

lags = list(range(1, 25)) + [28, 32, 36]

# Base columns to lag per group
groups = {
    'da':           ['da_total', 'da_congestion'],
    'rt':           ['rt_total', 'rt_congestion'],
    'wind_f':       ['spp_wind_total_forecast_f'],
    'load_f':       ['spp_load_total_forecast_f'],
    'loadwindgen':  ['spp_loadwindgen_actual_a', 'spp_loadwindgen_forecast_f'],
    'genoutage':    ['spp_genoutage_total_actual_a', 'spp_genoutage_total_forecast_f'],
    'zonal_wind_f': [c for c in df.columns if '_spp_res_zonal_wind_forecast' in c],
}

# Create lag columns grouped by node_num so lags don't bleed across nodes
for group, base_cols in groups.items():
    for col in base_cols:
        if col not in df.columns:
            continue
        for lag in lags:
            df[f'{col}_t{lag}'] = df.groupby('node_num')[col].shift(lag)

# Summary table: rows=groups, cols=lags, cell=sample lag column name
rows = []
for group, base_cols in groups.items():
    existing = [c for c in base_cols if c in df.columns]
    row = {'group': group, 'n_base': len(existing), 'base_columns': '  |  '.join(existing)}
    for lag in lags:
        row[f't-{lag}'] = f'{existing[0]}_t{lag}' if existing else ''
    rows.append(row)

summary = pd.DataFrame(rows).set_index('group')
lag_labels = [f't-{l}' for l in lags]

pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_columns', None)
display(summary[['n_base', 'base_columns'] + lag_labels])

# ── Reorder: group matching columns together, keep original order within each group
group_keywords = [
    lambda c: c.startswith('da_') or c.endswith('_da_total') or c.endswith('_da_congestion'),
    lambda c: c.startswith('rt_') or c.endswith('_rt_total') or c.endswith('_rt_congestion'),
    lambda c: 'wind' in c and 'loadwindgen' not in c,
    lambda c: 'load' in c and 'loadwindgen' not in c,
    lambda c: 'loadwindgen' in c,
    lambda c: 'genoutage' in c,
]

id_cols = [c for c in ['dtHr', 'dt', 'hr', 'node_num'] if c in df.columns]
ordered_cols = id_cols.copy()
used = set(id_cols)

for match in group_keywords:
    block = [c for c in df.columns if c not in used and match(c)]
    ordered_cols += block
    used.update(block)

ordered_cols += [c for c in df.columns if c not in used]
df = df[ordered_cols]

# ── Summary table ─────────────────────────────────────────────────────────────
rows = []
for group, base_cols in groups.items():
    existing = [c for c in base_cols if c in df.columns]
    row = {'group': group, 'n_base': len(existing), 'base_columns': '  |  '.join(existing)}
    for lag in lags:
        row[f't-{lag}'] = f'{existing[0]}_t{lag}' if existing else ''
    rows.append(row)

summary = pd.DataFrame(rows).set_index('group')
lag_labels = [f't-{l}' for l in lags]
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.max_columns', None)
display(summary[['n_base', 'base_columns'] + lag_labels])
print(f'\ndf shape: {df.shape}  |  unique cols: {len(set(df.columns))}')
print('First 20 columns:', df.columns[:20].tolist())


/tmp/ipykernel_1468132/1182776487.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_t{lag}'] = df.groupby('node_num')[col].shift(lag)
/tmp/ipykernel_1468132/1182776487.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f'{col}_t{lag}'] = df.groupby('node_num')[col].shift(lag)
/tmp/ipykernel_1468132/1182776487.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.conc

,n_base,base_columns,t-1,t-2,t-3,t-4,t-5,t-6,t-7,t-8,t-9,t-10,t-11,t-12,t-13,t-14,t-15,t-16,t-17,t-18,t-19,t-20,t-21,t-22,t-23,t-24,t-28,t-32,t-36
group,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
da,2,da_total | da_congestion,da_total_t1,da_total_t2,da_total_t3,da_total_t4,da_total_t5,da_total_t6,da_total_t7,da_total_t8,da_total_t9,da_total_t10,da_total_t11,da_total_t12,da_total_t13,da_total_t14,da_total_t15,da_total_t16,da_total_t17,da_total_t18,da_total_t19,da_total_t20,da_total_t21,da_total_t22,da_total_t23,da_total_t24,da_total_t28,da_total_t32,da_total_t36
rt,2,rt_total | rt_congestion,rt_total_t1,rt_total_t2,rt_total_t3,rt_total_t4,rt_total_t5,rt_total_t6,rt_total_t7,rt_total_t8,rt_total_t9,rt_total_t10,rt_total_t11,rt_total_t12,rt_total_t13,rt_total_t14,rt_total_t15,rt_total_t16,rt_total_t17,rt_total_t18,rt_total_t19,rt_total_t20,rt_total_t21,rt_total_t22,rt_total_t23,rt_total_t24,rt_total_t28,rt_total_t32,rt_total_t36
wind_f,1,spp_wind_total_forecast_f,spp_wind_total_forecast_f_t1,spp_wind_total_forecast_f_t2,spp_wind_total_forecast_f_t3,spp_wind_total_forecast_f_t4,spp_wind_total_forecast_f_t5,spp_wind_total_forecast_f_t6,spp_wind_total_forecast_f_t7,spp_wind_total_forecast_f_t8,spp_wind_total_forecast_f_t9,spp_wind_total_forecast_f_t10,spp_wind_total_forecast_f_t11,spp_wind_total_forecast_f_t12,spp_wind_total_forecast_f_t13,spp_wind_total_forecast_f_t14,spp_wind_total_forecast_f_t15,spp_wind_total_forecast_f_t16,spp_wind_total_forecast_f_t17,spp_wind_total_forecast_f_t18,spp_wind_total_forecast_f_t19,spp_wind_total_forecast_f_t20,spp_wind_total_forecast_f_t21,spp_wind_total_forecast_f_t22,spp_wind_total_forecast_f_t23,spp_wind_total_forecast_f_t24,spp_wind_total_forecast_f_t28,spp_wind_total_forecast_f_t32,spp_wind_total_forecast_f_t36
load_f,1,spp_load_total_forecast_f,spp_load_total_forecast_f_t1,spp_load_total_forecast_f_t2,spp_load_total_forecast_f_t3,spp_load_total_forecast_f_t4,spp_load_total_forecast_f_t5,spp_load_total_forecast_f_t6,spp_load_total_forecast_f_t7,spp_load_total_forecast_f_t8,spp_load_total_forecast_f_t9,spp_load_total_forecast_f_t10,spp_load_total_forecast_f_t11,spp_load_total_forecast_f_t12,spp_load_total_forecast_f_t13,spp_load_total_forecast_f_t14,spp_load_total_forecast_f_t15,spp_load_total_forecast_f_t16,spp_load_total_forecast_f_t17,spp_load_total_forecast_f_t18,spp_load_total_forecast_f_t19,spp_load_total_forecast_f_t20,spp_load_total_forecast_f_t21,spp_load_total_forecast_f_t22,spp_load_total_forecast_f_t23,spp_load_total_forecast_f_t24,spp_load_total_forecast_f_t28,spp_load_total_forecast_f_t32,spp_load_total_forecast_f_t36
loadwindgen,2,spp_loadwindgen_actual_a | spp_loadwindgen_f...,spp_loadwindgen_actual_a_t1,spp_loadwindgen_actual_a_t2,spp_loadwindgen_actual_a_t3,spp_loadwindgen_actual_a_t4,spp_loadwindgen_actual_a_t5,spp_loadwindgen_actual_a_t6,spp_loadwindgen_actual_a_t7,spp_loadwindgen_actual_a_t8,spp_loadwindgen_actual_a_t9,spp_loadwindgen_actual_a_t10,spp_loadwindgen_actual_a_t11,spp_loadwindgen_actual_a_t12,spp_loadwindgen_actual_a_t13,spp_loadwindgen_actual_a_t14,spp_loadwindgen_actual_a_t15,spp_loadwindgen_actual_a_t16,spp_loadwindgen_actual_a_t17,spp_loadwindgen_actual_a_t18,spp_loadwindgen_actual_a_t19,spp_loadwindgen_actual_a_t20,spp_loadwindgen_actual_a_t21,spp_loadwindgen_actual_a_t22,spp_loadwindgen_actual_a_t23,spp_loadwindgen_actual_a_t24,spp_loadwindgen_actual_a_t28,spp_loadwindgen_actual_a_t32,spp_loadwindgen_actual_a_t36
genoutage,2,spp_genoutage_total_actual_a | spp_genoutage...,spp_genoutage_total_actual_a_t1,spp_genoutage_total_actual_a_t2,spp_genoutage_total_actual_a_t3,spp_genoutage_total_actual_a_t4,spp_genoutage_total_actual_a_t5,spp_genoutage_total_actual_a_t6,spp_genoutage_total_actual_a_t7,spp_genoutage_total_actual_a_t8,spp_genoutage_total_actual_a_t9,spp_genoutage_total_actual_a_t10,spp_genoutage_total_actual_a_t11,spp_genoutage_total_actual_a_t12,spp_genoutage_total_actual_a_t13,spp_genoutage_total_actual_a_t14,spp_genoutage_total_actual_a_t15,spp_genoutage_total_

,n_base,base_columns,t-1,t-2,t-3,t-4,t-5,t-6,t-7,t-8,t-9,t-10,t-11,t-12,t-13,t-14,t-15,t-16,t-17,t-18,t-19,t-20,t-21,t-22,t-23,t-24,t-28,t-32,t-36
group,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
da,2,da_total | da_congestion,da_total_t1,da_total_t2,da_total_t3,da_total_t4,da_total_t5,da_total_t6,da_total_t7,da_total_t8,da_total_t9,da_total_t10,da_total_t11,da_total_t12,da_total_t13,da_total_t14,da_total_t15,da_total_t16,da_total_t17,da_total_t18,da_total_t19,da_total_t20,da_total_t21,da_total_t22,da_total_t23,da_total_t24,da_total_t28,da_total_t32,da_total_t36
rt,2,rt_total | rt_congestion,rt_total_t1,rt_total_t2,rt_total_t3,rt_total_t4,rt_total_t5,rt_total_t6,rt_total_t7,rt_total_t8,rt_total_t9,rt_total_t10,rt_total_t11,rt_total_t12,rt_total_t13,rt_total_t14,rt_total_t15,rt_total_t16,rt_total_t17,rt_total_t18,rt_total_t19,rt_total_t20,rt_total_t21,rt_total_t22,rt_total_t23,rt_total_t24,rt_total_t28,rt_total_t32,rt_total_t36
wind_f,1,spp_wind_total_forecast_f,spp_wind_total_forecast_f_t1,spp_wind_total_forecast_f_t2,spp_wind_total_forecast_f_t3,spp_wind_total_forecast_f_t4,spp_wind_total_forecast_f_t5,spp_wind_total_forecast_f_t6,spp_wind_total_forecast_f_t7,spp_wind_total_forecast_f_t8,spp_wind_total_forecast_f_t9,spp_wind_total_forecast_f_t10,spp_wind_total_forecast_f_t11,spp_wind_total_forecast_f_t12,spp_wind_total_forecast_f_t13,spp_wind_total_forecast_f_t14,spp_wind_total_forecast_f_t15,spp_wind_total_forecast_f_t16,spp_wind_total_forecast_f_t17,spp_wind_total_forecast_f_t18,spp_wind_total_forecast_f_t19,spp_wind_total_forecast_f_t20,spp_wind_total_forecast_f_t21,spp_wind_total_forecast_f_t22,spp_wind_total_forecast_f_t23,spp_wind_total_forecast_f_t24,spp_wind_total_forecast_f_t28,spp_wind_total_forecast_f_t32,spp_wind_total_forecast_f_t36
load_f,1,spp_load_total_forecast_f,spp_load_total_forecast_f_t1,spp_load_total_forecast_f_t2,spp_load_total_forecast_f_t3,spp_load_total_forecast_f_t4,spp_load_total_forecast_f_t5,spp_load_total_forecast_f_t6,spp_load_total_forecast_f_t7,spp_load_total_forecast_f_t8,spp_load_total_forecast_f_t9,spp_load_total_forecast_f_t10,spp_load_total_forecast_f_t11,spp_load_total_forecast_f_t12,spp_load_total_forecast_f_t13,spp_load_total_forecast_f_t14,spp_load_total_forecast_f_t15,spp_load_total_forecast_f_t16,spp_load_total_forecast_f_t17,spp_load_total_forecast_f_t18,spp_load_total_forecast_f_t19,spp_load_total_forecast_f_t20,spp_load_total_forecast_f_t21,spp_load_total_forecast_f_t22,spp_load_total_forecast_f_t23,spp_load_total_forecast_f_t24,spp_load_total_forecast_f_t28,spp_load_total_forecast_f_t32,spp_load_total_forecast_f_t36
loadwindgen,2,spp_loadwindgen_actual_a | spp_loadwindgen_f...,spp_loadwindgen_actual_a_t1,spp_loadwindgen_actual_a_t2,spp_loadwindgen_actual_a_t3,spp_loadwindgen_actual_a_t4,spp_loadwindgen_actual_a_t5,spp_loadwindgen_actual_a_t6,spp_loadwindgen_actual_a_t7,spp_loadwindgen_actual_a_t8,spp_loadwindgen_actual_a_t9,spp_loadwindgen_actual_a_t10,spp_loadwindgen_actual_a_t11,spp_loadwindgen_actual_a_t12,spp_loadwindgen_actual_a_t13,spp_loadwindgen_actual_a_t14,spp_loadwindgen_actual_a_t15,spp_loadwindgen_actual_a_t16,spp_loadwindgen_actual_a_t17,spp_loadwindgen_actual_a_t18,spp_loadwindgen_actual_a_t19,spp_loadwindgen_actual_a_t20,spp_loadwindgen_actual_a_t21,spp_loadwindgen_actual_a_t22,spp_loadwindgen_actual_a_t23,spp_loadwindgen_actual_a_t24,spp_loadwindgen_actual_a_t28,spp_loadwindgen_actual_a_t32,spp_loadwindgen_actual_a_t36
genoutage,2,spp_genoutage_total_actual_a | spp_genoutage...,spp_genoutage_total_actual_a_t1,spp_genoutage_total_actual_a_t2,spp_genoutage_total_actual_a_t3,spp_genoutage_total_actual_a_t4,spp_genoutage_total_actual_a_t5,spp_genoutage_total_actual_a_t6,spp_genoutage_total_actual_a_t7,spp_genoutage_total_actual_a_t8,spp_genoutage_total_actual_a_t9,spp_genoutage_total_actual_a_t10,spp_genoutage_total_actual_a_t11,spp_genoutage_total_actual_a_t12,spp_genoutage_total_actual_a_t13,spp_genoutage_total_actual_a_t14,spp_genoutage_total_actual_a_t15,spp_genoutage_total_


df shape: (17590, 611)  |  unique cols: 611
First 20 columns: ['dtHr', 'dt', 'hr', 'node_num', 'da_total', 'da_congestion', 'da_slack', 'da_totalMinus1D', 'da_congestionMinus1D', 'da_totalHistory', 'SPPSimHr_da_total', 'SPPSimHr_da_congestion', 'SPPResZone4SimHr_da_total', 'SPPResZone4SimHr_da_congestion', 'da_total_t1', 'da_total_t2', 'da_total_t3', 'da_total_t4', 'da_total_t5', 'da_total_t6']


In [59]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import math
import time
import numpy as np
import pandas as pd
from nighthawk.models.valuation import ve_model_functions

def GRU_framework_reorder(node_num, dt):

    run_number    = 1
    opexchange    = 'SPP'

    if int(run_number) == 1:
        data_location = 'training'
    else:
        data_location = 'secondRun'

    opexchange     = "SPP"
    data_location  = "training"
    gs_loc         = f"gs://ve_fourier/production/SPP/{data_location}"

    max_retries = 1
    retry_delay = 3
    for attempt in range(max_retries):
        try:
            data_df = pd.read_csv(f"{gs_loc}/{node_num}_{dt}.csv")
            data_df["dt"] = pd.to_datetime(data_df["dt"]).dt.strftime("%Y-%m-%d")
            break
        except Exception as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            else:
                raise

    selected_columns = [col for col in data_df.columns if "iirGen" not in col and "txoutage" not in col and "topGen" not in col]
    data_df = data_df[selected_columns]


    feb2021_dates = pd.DataFrame(pd.date_range("2021-02-13", "2021-02-19", freq="D"), columns=["dt"])["dt"].dt.strftime("%Y-%m-%d").tolist()
    if dt not in feb2021_dates:
        data_df = data_df[~data_df["dt"].isin(feb2021_dates)]

    if "dtHr" not in data_df.columns:
        data_df.insert(0, column="dtHr", value=pd.to_datetime(data_df["dt"]) + pd.to_timedelta(data_df["hr"] - 1, unit="h"))

    lags = list(range(1, 25)) + [28, 32, 36]

    # Base columns to lag per group
    groups = {
        'da':           ['da_total', 'da_congestion'],
        'rt':           ['rt_total', 'rt_congestion'],
        'wind_f':       ['spp_wind_total_forecast_f'],
        'load_f':       ['spp_load_total_forecast_f'],
        'loadwindgen':  ['spp_loadwindgen_actual_a', 'spp_loadwindgen_forecast_f'],
        'genoutage':    ['spp_genoutage_total_actual_a', 'spp_genoutage_total_forecast_f'],
        'zonal_wind_f': [c for c in data_df.columns if '_spp_res_zonal_wind_forecast' in c],
    }

    # Create lag columns grouped by node_num so lags don't bleed across nodes
    for group, base_cols in groups.items():
        for col in base_cols:
            if col not in data_df.columns:
                continue
            for lag in lags:
                data_df[f'{col}_t{lag}'] = data_df.groupby('node_num')[col].shift(lag)


    # ── Reorder: group matching columns together, keep original order within each group
    group_keywords = [
        lambda c: c.startswith('da_') or c.endswith('_da_total') or c.endswith('_da_congestion'),
        lambda c: c.startswith('rt_') or c.endswith('_rt_total') or c.endswith('_rt_congestion'),
        lambda c: 'wind' in c and 'loadwindgen' not in c,
        lambda c: 'load' in c and 'loadwindgen' not in c,
        lambda c: 'loadwindgen' in c,
        lambda c: 'genoutage' in c,
    ]

    id_cols = [c for c in ['dtHr', 'dt', 'hr', 'node_num'] if c in data_df.columns]
    ordered_cols = id_cols.copy()
    used = set(id_cols)

    for match in group_keywords:
        block = [c for c in data_df.columns if c not in used and match(c)]
        ordered_cols += block
        used.update(block)

    ordered_cols += [c for c in data_df.columns if c not in used]
    data_df = data_df[ordered_cols]
    print(data_df.head())
    print(data_df.shape)

    # ── model classes (all inside function) ─────────────────────

    class GRUMean(nn.Module):
        """
        GRU-based network for mean prediction.
        Treats the flat feature vector as a length-input_size sequence with 1 feature per step,
        runs a bidirectional multi-layer GRU, then projects the final hidden state to outputs.
        """
        def __init__(self, input_size, output_size, hidden_size=64, num_layers=2,
                bidirectional=True, dropout=0.1):
            super(GRUMean, self).__init__()
            self.hidden_size   = hidden_size
            self.num_layers    = num_layers
            self.bidirectional = bidirectional
            self.gru = nn.GRU(
                input_size=1,
                hidden_size=hidden_size,
                num_layers=num_layers,
                batch_first=True,
                bidirectional=bidirectional,
                dropout=dropout if num_layers > 1 else 0.0,
            )
            out_dim = hidden_size * (2 if bidirectional else 1)
            # Attention pooling layer — produces a scalar weight per sequence position
            self.attention = nn.Linear(out_dim, 1)
            self.fc = nn.Linear(out_dim, output_size)

        def forward(self, x):
            # x shape: (B, input_size)
            x = x.unsqueeze(-1)                                # (B, seq_len, 1)
            out, _ = self.gru(x)                               # (B, seq_len, hidden*dirs)
            # Attention pooling: learned weighted sum of positions
            attn_logits = self.attention(out)                  # (B, seq_len, 1)
            attn_weights = torch.softmax(attn_logits, dim=1)   # (B, seq_len, 1)
            pooled = (out * attn_weights).sum(dim=1)           # (B, hidden*dirs)
            return self.fc(pooled)   

    def predict_and_collect(model, data_loader, criterion, scaler_y=None):
        model.eval()
        total_loss = 0.0
        predictions = []
        with torch.no_grad():
            for inputs, labels in data_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                total_loss += criterion(outputs, labels).item()
                predictions.append(outputs.cpu().numpy())
        predictions = np.concatenate(predictions, axis=0)
        if normalize and scaler_y is not None:
            predictions = scaler_y.inverse_transform(predictions)
        return predictions, total_loss / len(data_loader)


    # ── hyper-parameters ────────────────────────────────────────
    learning_rate  = 0.001
    num_epochs     = 50
    normalize      = True
    device         = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ── GRU hyper-parameters ────────────────────────────────────
    hidden_size   = 16
    num_layers    = 2
    bidirectional = True
    dropout       = 0.2

    mean_criterion = nn.HuberLoss(delta = 1)

    valuation_models = pd.DataFrame()
    
    for y_var in [["da_total", "rt_total"]]:
        trainingPeriod = int(min((pd.to_datetime(dt) - pd.to_datetime("2017-01-01")) / np.timedelta64(1, "D"), 730))
        xtrain, ytrain, xtest, ytest = ve_model_functions.getTrainTestData(
            opexchange, data_df, y=y_var, bidDt=dt, hour_gap=18,
            trainingPeriod=str(trainingPeriod) + "D", train_a_or_f="f")

        # 3-way split: 1/2 train, 1/4 validate, 1/4 test
        xtrain, xtest, ytrain, ytest = train_test_split(xtrain, ytrain, test_size=1/8, shuffle=False)
        xtrain, xvalidate, ytrain, yvalidate = train_test_split(xtrain, ytrain, test_size=1/7, shuffle=False)
        scaler_x, scaler_y = None, None
        if normalize:
            scaler_x = StandardScaler()
            scaler_y = StandardScaler()
            xtrain    = pd.DataFrame(scaler_x.fit_transform(xtrain),    columns=xtrain.columns,    index=xtrain.index)
            xvalidate = pd.DataFrame(scaler_x.transform(xvalidate),     columns=xvalidate.columns, index=xvalidate.index)
            xtest     = pd.DataFrame(scaler_x.transform(xtest),         columns=xtest.columns,     index=xtest.index)
            ytrain    = pd.DataFrame(scaler_y.fit_transform(ytrain),    columns=ytrain.columns,    index=ytrain.index)
            yvalidate = pd.DataFrame(scaler_y.transform(yvalidate),     columns=yvalidate.columns, index=yvalidate.index)
            ytest     = pd.DataFrame(scaler_y.transform(ytest),         columns=ytest.columns,     index=ytest.index)
        X_train_tensor    = torch.tensor(xtrain.values,    dtype=torch.float32).to(device)
        Y_train_tensor    = torch.tensor(ytrain.values,    dtype=torch.float32).to(device)
        X_validate_tensor = torch.tensor(xvalidate.values, dtype=torch.float32).to(device)
        Y_validate_tensor = torch.tensor(yvalidate.values, dtype=torch.float32).to(device)
        X_test_tensor     = torch.tensor(xtest.values,     dtype=torch.float32).to(device)
        Y_test_tensor     = torch.tensor(ytest.values,     dtype=torch.float32).to(device)

        g = torch.Generator()
        g.manual_seed(42)
        train_loader    = DataLoader(TensorDataset(X_train_tensor, Y_train_tensor),       batch_size=128, shuffle=True, generator=g)
        validate_loader = DataLoader(TensorDataset(X_validate_tensor, Y_validate_tensor), batch_size=128, shuffle=False)
        test_loader     = DataLoader(TensorDataset(X_test_tensor, Y_test_tensor),         batch_size=128, shuffle=False)
        input_size, output_size = X_train_tensor.shape[1], Y_train_tensor.shape[1]

        # ── build model ─────────────────────────────────────────
        mean_model = GRUMean(input_size, output_size,
                             hidden_size=hidden_size, num_layers=num_layers,
                             bidirectional=bidirectional, dropout=dropout).to(device)
        mean_optimizer = optim.AdamW(mean_model.parameters(), lr=learning_rate, weight_decay=1e-4)
        mean_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(mean_optimizer, T_max=num_epochs, eta_min=1e-6)

        # ── train mean model ────────────────────────────────────
        best_val_loss, patience, no_improve = float("inf"), 5, 0
        for epoch in range(num_epochs):
            mean_model.train()
            for inputs, labels in train_loader:
                mean_optimizer.zero_grad()
                loss = mean_criterion(mean_model(inputs), labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(mean_model.parameters(), max_norm=1.0)  # Add this line
                mean_optimizer.step()
            mean_model.eval()
            val_loss = sum(mean_criterion(mean_model(inp), lbl).item() for inp, lbl in validate_loader) / len(validate_loader)
            print(f"Epoch {epoch + 1}/{num_epochs} | val_loss={val_loss:.4f}")
            if val_loss < best_val_loss:
                best_val_loss, no_improve = val_loss, 0
            else:
                no_improve += 1
                if no_improve >= patience:
                    print(f"Early stopping at epoch {epoch + 1}")
                    break
            mean_scheduler.step()

        mean_preds, _ = predict_and_collect(mean_model, test_loader, mean_criterion, scaler_y)

        if normalize and scaler_y is not None:
            ytest = pd.DataFrame(scaler_y.inverse_transform(ytest), columns=ytest.columns, index=ytest.index)

        mean_df = pd.DataFrame(mean_preds, columns=[f"{y}_mean" for y in y_var])

        result = pd.concat([ytest.reset_index(), mean_df], axis=1)
        result["dtHr"] = pd.to_datetime(result["dtHr"])
        result["dt"]   = result["dtHr"].dt.strftime("%Y-%m-%d")
        result["hr"]   = result["dtHr"].dt.hour + 1
        valuation_models = pd.concat([valuation_models, result])

    valuation_models = valuation_models.groupby(["dt", "hr", "node_num"]).max().reset_index()
    keep_cols = ["dt", "hr", "node_num", "da_total_mean", "rt_total_mean"]
    direction_tag = "bi" if bidirectional else "uni"
    valuation_models["model"] = f"gru_{direction_tag}_{hidden_size}h_{num_layers}L_mean_only"

    return valuation_models

In [60]:
result = GRU_framework_reorder(144, '2026-02-13')

                  dtHr          dt  hr  node_num  da_total  da_congestion  \
0  2024-02-13 00:00:00  2024-02-13   1       144   23.8935        -1.5114   
1  2024-02-13 01:00:00  2024-02-13   2       144   21.2625        -1.5061   
2  2024-02-13 02:00:00  2024-02-13   3       144   20.4319        -1.7013   
3  2024-02-13 03:00:00  2024-02-13   4       144   21.3000         0.2767   
4  2024-02-13 04:00:00  2024-02-13   5       144   22.3000        -0.5379   

   da_slack  da_totalMinus1D  da_congestionMinus1D  da_totalHistory  \
0   25.2290          18.7178               -0.1699          18.7178   
1   22.5481          15.4701                0.0136          15.4701   
2   21.7505          14.5872                0.0264          14.5872   
3   20.5460          15.1955               -0.1050          15.1955   
4   22.2373          17.7972               -0.4423          17.7972   

   SPPSimHr_da_total  SPPSimHr_da_congestion  SPPResZone4SimHr_da_total  \
0            20.1934               

KeyboardInterrupt: 